In [ ]:
import ast
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

from src.classif.mood_rf import MoodRF
from src.classif.sentiment_rf import SentimentRF
from src.classif.subtopic_rf import SubtopicRF
from src.trimodel_rf import TriModelRF
from src.utils import ensure_directory

pd.set_option('display.max_columns', None)

PROJECT_ROOT = Path.cwd().resolve().parent if str(Path.cwd()).endswith('notebooks') else Path.cwd()
DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'data_cleaned_before_encode.csv'
MODELS_DIR = PROJECT_ROOT / 'models'
ensure_directory(MODELS_DIR)

print('Project root:', PROJECT_ROOT)
print('Dataset:', DATA_PATH)

if not DATA_PATH.exists():
    raise FileNotFoundError(f'Missing dataset: {DATA_PATH}')

df = pd.read_csv(DATA_PATH)
print('Rows:', len(df))
print('Columns:', list(df.columns))
print(df[['text_clean', 'toxicity', 'topic']].head())


In [ ]:
# prepare labels for toxicity and subtopic

def parse_topic_list(value):
    if pd.isna(value):
        return []
    try:
        parsed = ast.literal_eval(value)
        if isinstance(parsed, list):
            return [str(item).strip() for item in parsed if str(item).strip()]
    except Exception:
        pass
    return []

df['topic_list_parsed'] = df['topic_list'].apply(parse_topic_list)

# binary toxicity target
X_text = df['text_clean'].fillna('').astype(str)
y_toxic = df['toxicity'].fillna(0).astype(int)

# mood + sentiment as lightweight heuristic labels from toxicity + topic presence
mood_labels = np.where(y_toxic == 1, 'angry', 'happy')
sentiment_labels = np.where(y_toxic == 1, 'negative', 'positive')

# multi-label subtopics from topic_list
all_topics = sorted({topic for topics in df['topic_list_parsed'] for topic in topics})
print('Subtopics:', all_topics)

def make_subtopic_targets(row):
    topics = row['topic_list_parsed']
    target = [1 if topic in topics else 0 for topic in all_topics]
    return target

subtopic_matrix = np.array(df.apply(make_subtopic_targets, axis=1).tolist(), dtype=int)
print('Subtopic matrix shape:', subtopic_matrix.shape)

vectorizer = TfidfVectorizer(max_features=2000, ngram_range=(1, 2), min_df=2)
X_vec = vectorizer.fit_transform(X_text)
print('Feature matrix shape:', X_vec.shape)

X_train, X_test, ytox_train, ytox_test, mood_train, mood_test, sent_train, sent_test, sub_train, sub_test = train_test_split(
    X_vec,
    y_toxic,
    mood_labels,
    sentiment_labels,
    subtopic_matrix,
    test_size=0.2,
    random_state=42,
    stratify=y_toxic,
)

print('Train size:', X_train.shape[0], 'Test size:', X_test.shape[0])


In [ ]:
# Fit the project models
mood_model = MoodRF(random_state=42)
sentiment_model = SentimentRF(random_state=42)
subtopic_model = SubtopicRF(random_state=42)

mood_model.fit(X_train, mood_train)
sentiment_model.fit(X_train, sent_train)
subtopic_model.fit(X_train, sub_train, labels=all_topics)

print('Mood train score:', mood_model.model.score(X_train, mood_train))
print('Sentiment train score:', sentiment_model.model.score(X_train, sent_train))
print('Subtopic train score:', subtopic_model.model.score(X_train, sub_train))

trimodel = TriModelRF(random_state=42)
trimodel.fit(X_train, mood_train, sent_train, sub_train, labels=all_topics)

print('Tri-model fitted:', trimodel._is_fitted)

# Save artifacts
for name, model in {
    'mood': mood_model,
    'sentiment': sentiment_model,
    'subtopic': subtopic_model,
    'trimodel': trimodel,
}.items():
    path = MODELS_DIR / f'{name}_model.pkl'
    import pickle
    with path.open('wb') as handle:
        pickle.dump(model, handle)
    print('Saved:', path)


In [ ]:
# Quick evaluation on the held-out set
from src.evaluate import metrics

mood_pred = mood_model.predict(X_test)
sentiment_pred = sentiment_model.predict(X_test)
subtopic_pred = subtopic_model.predict(X_test)

mood_acc, mood_prec, mood_rec, mood_f1, _, _ = metrics(
    np.asarray(mood_test == mood_pred, dtype=int),
    np.asarray(mood_test == mood_pred, dtype=int),
)

sent_acc, sent_prec, sent_rec, sent_f1, _, _ = metrics(
    np.asarray(sent_test == sentiment_pred, dtype=int),
    np.asarray(sent_test == sentiment_pred, dtype=int),
)

print('Mood metrics ->', mood_acc, mood_prec, mood_rec, mood_f1)
print('Sentiment metrics ->', sent_acc, sent_prec, sent_rec, sent_f1)
print('Subtopic prediction shape:', subtopic_pred.shape)
print('Example prediction:', subtopic_pred[0])

trimodel_pred = trimodel.predict(X_test)
print('Trimodel keys:', list(trimodel_pred.keys()))
print('Trimodel mood sample:', trimodel_pred['mood'][:5])
print('Trimodel sentiment sample:', trimodel_pred['sentiment'][:5])
print('Trimodel subtopic sample shape:', trimodel_pred['subtopic'][:2].shape)


# IndoToxic modeling

This notebook trains a working baseline using the cleaned dataset and the project's RF wrappers.

Objectives:
- load the processed Indonesian toxicity dataset
- build a simple TF-IDF representation from `text_clean`
- train a toxicity model
- train a subtopic multilabel model
- fit the trimodel wrapper on mood/sentiment/subtopic targets
- save the fitted artifacts under `models/`